## Custom Transformers

A custom transformer `CustomScaler` is defined for z-score scaling (standardizing)
of data. Since z-score scaling needs the feature-wise **mean** and **standard
deviation** learned from the data, the `fit` method computes and stores these as
`self.mean_` and `self.std_`. The `transform` method then applies the z-score
formula using those stored, per-feature statistics — so a test set is scaled
consistently with the training set it was fit on, rather than recomputing its
own statistics.

# `fit_transform` vs `fit_predict` in scikit-learn

Both fit an estimator and apply it to the same data in one call. They differ in which estimators have them and what they return.

## `fit_transform(X)`

- **Used by:** transformers, such as `StandardScaler`, `PCA`, `OneHotEncoder`, `TfidfVectorizer`, and `TSNE`
- **What it does:** learns parameters from `X` (means/variances, principal components, vocabulary) and returns a **transformed feature matrix**
- **Equivalent to:** `fit(X).transform(X)`, sometimes computed more efficiently in one pass
- **Typical workflow:** fit on training data, then call `transform(X_new)` on test data to avoid leakage

## `fit_predict(X)`

- **Used by:** clustering and outlier-detection estimators, such as `KMeans`, `DBSCAN`, `GaussianMixture`, `IsolationForest`, and `LocalOutlierFactor`
- **What it does:** fits the model and returns a **label per sample**, either cluster assignments or inlier/outlier flags (`1` / `-1`)
- **Equivalent to:** `fit(X).predict(X)`, where `predict` exists

## Subtleties

- **Transductive methods have only the combined call.**
  - `DBSCAN` and `AgglomerativeClustering` have `fit_predict` but no `predict`, because their labels come from the fitted data's structure and there's no natural rule for new points.
  - `TSNE` has `fit_transform` but no `transform`, because the embedding is optimized for the given points rather than learned as a reusable mapping.
- `LocalOutlierFactor` exposes `predict` only when `novelty=True`.
- **Supervised models** (classifiers and regressors) generally don't provide `fit_predict`, since predicting on the same labeled training data isn't a standard workflow.

## Summary

| | `fit_transform` | `fit_predict` |
|---|---|---|
| Estimator type | Transformers | Clusterers / outlier detectors |
| Returns | Transformed features `(n_samples, n_features')` | Labels `(n_samples,)` |
| Apply to new data | `transform(X_new)` | `predict(X_new)`, if available |

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
# BaseEstimator gives the class standard scikit-learn plumbing (like
# get_params()/set_params(), used internally by tools such as
# GridSearchCV). TransformerMixin automatically adds a fit_transform()
# method for free, built by calling fit() then transform() in
# sequence — so this custom class immediately gets the same
# fit/transform/fit_transform interface used by every other
# transformer throughout this series (CountVectorizer, StandardScaler,
# PCA, etc.), just by inheriting from these two.

import numpy as np
# Needed for np.mean / np.std below.

class CustomScaler(BaseEstimator, TransformerMixin):
      def fit(self, X, y=None):
          self.mean_ = X.mean(axis=0)
          self.std_ = X.std(axis=0)
          return self
    # As before: fit() does no work and stores nothing — just returns
    # self so fit_transform() (from TransformerMixin) has something to
    # chain into transform() afterward. y=None is accepted purely for
    # interface compatibility (scikit-learn's fit() convention always
    # accepts an optional y, even for unsupervised transformers that
    # ignore it, like PCA's own fit()).

      def transform(self, X, y=None):
           return (X - self.mean_) / self.std_  # simple z-score scaling
        # Applies the z-score formula using the per-column mean and std
    # that fit() already computed and stored (self.mean_, self.std_),
    # rather than recomputing them from X. This is what makes the
    # transformer correct: new data (e.g. a test set) gets scaled
    # relative to the TRAINING set's statistics, consistent with how
    # StandardScaler and every other fit/transform transformer in this
    # series behaves — and since self.mean_/self.std_ were computed
    # with axis=0, each feature is scaled independently rather than
    # mixed into one global pool.

In [ ]:
custom_scaler = CustomScaler()
# Instantiate the CustomScaler class defined in the previous cell,
# creating a usable object — no arguments needed here, since the
# class's __init__ wasn't overridden (it just uses the default,
# empty constructor inherited from BaseEstimator/TransformerMixin).
#
# At this point, nothing has been fit or transformed yet — this line
# only creates the object, ready to be used like any other scikit-learn
# transformer, e.g.:
#   custom_scaler.fit(X_train)
#   X_train_scaled = custom_scaler.transform(X_train)
# or in one step:
#   X_train_scaled = custom_scaler.fit_transform(X_train)
#   (this fit_transform() comes for free from TransformerMixin, calling
#   fit() then transform() internally, as noted earlier)

In [ ]:
from sklearn.datasets import load_iris
# Import scikit-learn's loader for the classic Iris dataset — the same
# built-in dataset used earlier in this series (PCA, MDS, LLE, Isomap,
# t-SNE) for 150 flower samples with 4 continuous measurements each.

data = load_iris()
# Load the dataset into `data`, a scikit-learn Bunch object bundling
# the feature matrix (data.data, shape (150, 4)), the species labels
# (data.target), and metadata (data.feature_names, data.target_names)
# all together — same structure as the earlier `iris = datasets.load_iris()`
# calls, just assigned to a differently-named variable here (`data`
# instead of `iris`).


In [ ]:
X = data["data"]
# Extract the feature matrix from the Bunch object using dict-style
# indexing — data["data"] works the same as the more common
# attribute-style data.data, since a scikit-learn Bunch supports both.
# X is a NumPy array of shape (150, 4): 150 samples x 4 features
# (sepal length, sepal width, petal length, petal width).
#
# Pulling X out as its own variable, separate from the labels, is what
# lets it be passed directly into CustomScaler: a transformer only
# operates on the feature matrix, so isolating X here is what
# fit()/transform() will expect as input in the next steps.

In [ ]:
y = data["target"]
# Extract the target labels from the Bunch object, again using
# dict-style indexing. y is a NumPy array of shape (150,), holding
# each sample's species as an integer code (0, 1, or 2).
#
# Separating y from X here keeps labels and features as two distinct
# variables, matching the (X, y) convention scikit-learn expects
# throughout: transformers like CustomScaler act only on X, while y
# is kept aside for later steps such as train/test splitting or
# training a classifier.

##You have to choose either one of them depending upon your objective

In [ ]:
# Two steps — useful when you want to inspect custom_scaler.mean_ /
# custom_scaler.std_ afterward, or reuse them on a separate test set:
custom_scaler.fit(X)
custom_scaler.transform(X)

#####OR###############

# One step — fit and transform in a single call, when there's no
# separate test set and no need to inspect the learned stats first:
# custom_scaler.fit_transform(X)

array([[-9.00681170e-01,  1.01900435e+00, -1.34022653e+00,
        -1.31544430e+00],
       [-1.14301691e+00, -1.31979479e-01, -1.34022653e+00,
        -1.31544430e+00],
       [-1.38535265e+00,  3.28414053e-01, -1.39706395e+00,
        -1.31544430e+00],
       [-1.50652052e+00,  9.82172869e-02, -1.28338910e+00,
        -1.31544430e+00],
       [-1.02184904e+00,  1.24920112e+00, -1.34022653e+00,
        -1.31544430e+00],
       [-5.37177559e-01,  1.93979142e+00, -1.16971425e+00,
        -1.05217993e+00],
       [-1.50652052e+00,  7.88807586e-01, -1.34022653e+00,
        -1.18381211e+00],
       [-1.02184904e+00,  7.88807586e-01, -1.28338910e+00,
        -1.31544430e+00],
       [-1.74885626e+00, -3.62176246e-01, -1.34022653e+00,
        -1.31544430e+00],
       [-1.14301691e+00,  9.82172869e-02, -1.28338910e+00,
        -1.44707648e+00],
       [-5.37177559e-01,  1.47939788e+00, -1.28338910e+00,
        -1.31544430e+00],
       [-1.26418478e+00,  7.88807586e-01, -1.22655167e+00,
      

## Interquartile Range (IQR)

- **IQR (Interquartile Range)** = Q3 − Q1, the spread of the middle 50% of the
  data (Q1 = 25th percentile, Q3 = 75th percentile).
- A common rule flags any value below **Q1 − 1.5×IQR** or above
  **Q3 + 1.5×IQR** as an outlier.
- Unlike `CustomScaler`, which transforms every value (rescales it) without
  changing the number of rows, `OutlierRemoval` will presumably **drop rows** —
  so its `transform()` needs to return an array with fewer samples than it was
  given, not just a rescaled version of the same shape.
- It will also need `fit()` to learn real parameters from the data —
  specifically Q1, Q3, and the IQR bounds — computed per column, the same way
  `CustomScaler` learns `self.mean_` / `self.std_`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
# Same base classes and import as CustomScaler — BaseEstimator for
# scikit-learn's standard plumbing, TransformerMixin for the free
# fit_transform(), and numpy for the percentile/median calculations
# below.

class OutlierRemoval(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        # Stores the IQR multiplier as a configurable parameter,
        # defaulting to 1.5 — the standard cutoff used in the common
        # IQR outlier rule. Making it a constructor argument (rather
        # than hardcoding 1.5 inside fit/transform) lets it be tuned,
        # e.g. OutlierRemoval(factor=3.0) for a stricter, more
        # tolerant cutoff that flags fewer points as outliers.

    def fit(self, X, y=None):
        self.median = np.median(X, axis=0)
        self.iqr = np.percentile(X, 75, axis=0) - np.percentile(X, 25, axis=0)
        return self
        # Computes two statistics per column (axis=0), same pattern as
        # CustomScaler's mean_/std_:
        #   - self.median: each feature's median value.
        #   - self.iqr: each feature's Q3 - Q1, the interquartile
        #     range, via two separate np.percentile calls.
        # Both are stored on the object so transform() can reuse them
        # without recomputing anything from scratch.

    def transform(self, X, y=None):
        lower_bound = self.median - self.factor * self.iqr
        upper_bound = self.median + self.factor * self.iqr
        return X[((X >= lower_bound) & (X <= upper_bound)).all(axis=1)]
        # Builds a lower and upper bound per feature, then keeps only
        # the ROWS where every single feature falls inside its bound:
        #   - (X >= lower_bound) & (X <= upper_bound): an element-wise
        #     boolean array, same shape as X, True wherever a value is
        #     within bounds for that feature.
        #   - .all(axis=1): collapses each row to a single True/False —
        #     True only if ALL features in that row are within bounds.
        #   - X[...]: boolean-indexes the rows, returning only the
        #     samples that passed on every feature. Any row with even
        #     one out-of-bounds feature is dropped entirely.


## Custom Classifiers


Here two custom classifiers are defined - `MajorityClassClassifier`, which
always predicts the majority class from the training data, and
`ThresholdClassifier`, which makes predictions based on a threshold on a
particular feature.

A few things worth having in mind before the code arrives:

- **Classifiers use `ClassifierMixin`, not `TransformerMixin`.** Instead of a
  free `fit_transform()`, `ClassifierMixin` gives a free `.score()` method
  (accuracy, by default) — since the goal now is predicting a label, not
  reshaping features.
- **`MajorityClassClassifier`'s `fit()` needs to look at `y`, not `X`.** Every
  transformer so far (`CustomScaler`, `OutlierRemoval`) only ever learned from
  the feature matrix. This is the first case where `fit()` actually needs the
  labels — it has to find whichever class appears most often in the training
  `y`, and store that single value.
- **Its `predict()` then ignores the input entirely.** Whatever `X` it's
  given, and however many rows it has, `predict()` should return that same
  stored majority class for every single row — a baseline that never actually
  looks at the features. This is a common sanity-check model: any real
  classifier should be expected to beat it.
- **`ThresholdClassifier` is to take a feature index and a threshold as
  constructor arguments** (the same pattern as `OutlierRemoval`'s `factor`),
  with `predict()` comparing that one feature's value against the threshold
  to decide the class — a rule-based classifier with no real learning in
  `fit()` at all.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.stats import mode
# ClassifierMixin (rather than TransformerMixin) marks this as a
# classifier, giving it a free .score() method for accuracy.
# scipy.stats.mode finds the most frequently occurring value in an
# array — exactly what's needed to find the majority class in y.

class MajorityClassClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.majority_class = mode(y)[0]
        return self
        # Finds the most common label in the training labels y and
        # stores it as self.majority_class.
        # mode(y) returns a ModeResult with two fields, mode and count;
        # [0] pulls out the mode value itself (the most frequent label),
        # discarding the count. On current scipy this value is already
        # a plain scalar, so no further indexing is needed.
        # X is accepted (scikit-learn's fit() signature requires it)
        # but never actually used — this classifier only ever looks
        # at y, never the features.

    def predict(self, X):
        return [self.majority_class] * len(X)
        # Ignores the actual values in X entirely, using only its
        # length: returns a list containing the same stored majority
        # class, repeated once per row in X. Every input, no matter
        # what it contains, gets the identical prediction.

## Inspecting the Data to Configure ThresholdClassifier

`ThresholdClassifier`'s `fit()` never learns anything from the data — `threshold`
and `feature_index` have to be chosen manually. To choose them well, we first
inspect the Iris features per species, looking for a feature where one class's
value range doesn't overlap with the others at all. A threshold placed inside
that gap will separate the classes perfectly; a threshold on an overlapping
feature never can, no matter what value is picked.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

data = load_iris()
# Load the same Iris dataset used earlier in this notebook.

X = data["data"]
y = data["target"]
# Pull out the feature matrix and integer species labels (0, 1, 2),
# matching the X, y already used by the custom transformers above.

df = pd.DataFrame(X, columns=data["feature_names"])
# Wrap X in a labeled DataFrame, purely for inspection — the
# classifier itself will still be called on the raw array X, not df.

df["species"] = pd.Categorical.from_codes(y, data["target_names"])
# Attach the species name to each row. This is what makes it possible
# to compare a feature's values ACROSS classes in the next step —
# y alone (just 0/1/2) doesn't let us group and describe by species
# without this extra column.

With the labeled DataFrame ready, check each feature's range within each
species. The feature to look for is one where a single cutoff value could
sit strictly between one species and the rest, with no overlap.

In [ ]:
df.groupby("species")["petal length (cm)"].describe()
# Group rows by species, then compute summary statistics (count, mean,
# std, min, max, quartiles) of petal length within each group.
# The min/max columns are what matter here: they show the full spread
# of petal length for each species, which is exactly what's needed to
# check for a non-overlapping gap between classes.

/tmp/ipykernel_751/3248756955.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("species")["petal length (cm)"].describe()


,count,mean,std,min,25%,50%,75%,max
species,,,,,,,,
setosa,50.0,1.462,0.173664,1.0,1.4,1.50,1.575,1.9
versicolor,50.0,4.260,0.469911,3.0,4.0,4.35,4.600,5.1
virginica,50.0,5.552,0.551895,4.5,5.1,5.55,5.875,6.9


The result shows setosa's petal length stays between 1.0 and 1.9 cm, while
versicolor and virginica both start at 3.0 cm or higher — a clean gap from
1.9 to 3.0 with no overlap at all. Any threshold inside that gap will
separate setosa from the other two species perfectly. This motivates the
`feature_index` and `threshold` used below: `feature_index=2` selects petal
length (`data["feature_names"][2]`), and `threshold=2.5` sits inside the gap
just found.

In [ ]:
class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    # Inherits from BaseEstimator (standard scikit-learn plumbing like
    # get_params()/set_params()) and ClassifierMixin (adds a free
    # .score() method, used later to compute accuracy).

    def __init__(self, threshold=0.5, feature_index=0):
        self.threshold = threshold
        self.feature_index = feature_index
        # Stores two configurable settings, not anything learned from
        # data:
        #   - threshold: the cutoff value a feature is compared against.
        #   - feature_index: WHICH column of X to look at (default 0,
        #     the first feature).
        # These are set when the object is created, e.g.
        # ThresholdClassifier(threshold=2.5, feature_index=2), and are
        # what the inspection step earlier was choosing sensible
        # values for.

    def fit(self, X, y):
        return self  # nothing to fit
        # This classifier's decision rule is fixed at construction
        # time (threshold + feature_index) rather than derived from
        # training data, so there's nothing for fit() to compute or
        # store. It just returns self, purely to satisfy scikit-learn's
        # expected fit/predict interface — every estimator is expected
        # to have a working fit() method, even one that does nothing.

    def predict(self, X):
        return (X[:, self.feature_index] > self.threshold).astype(int)
        # X[:, self.feature_index]: selects ONE column from X — every
        # row's value at that single feature — ignoring all other
        # columns.
        # (... > self.threshold): compares each of those values to
        # the stored threshold, producing a boolean array (True where
        # the feature exceeds it, False otherwise).

In [ ]:
threshold_clf = ThresholdClassifier(threshold=2.5, feature_index=2)
# Configure the classifier with the values motivated by the inspection
# above, rather than guessing them: petal length as the feature, and
# 2.5 as a cutoff that sits inside the observed 1.9-to-3.0 gap.

y_binary = (y != 0).astype(int)
# ThresholdClassifier's predict() only ever outputs 0 or 1 — it has no
# way to express 3 distinct classes. Recasting the label as "is this
# setosa (0) or not (1)" turns Iris's 3-class target into a binary one
# this classifier can actually be evaluated against.

With the classifier configured and the labels reframed as binary, fit it
(a no-op, kept only so the object follows the usual scikit-learn interface),
generate predictions, and check how many it got right.

In [ ]:
threshold_clf.fit(X, y_binary)
# No-op, as defined earlier — returns self without using X or y_binary.

predictions = threshold_clf.predict(X)
# Applies the rule "petal length > 2.5" to every row of X, producing
# a 0/1 prediction per sample.

accuracy = threshold_clf.score(X, y_binary)
# ClassifierMixin's built-in score(): runs predict(X) internally and
# compares the result to y_binary, returning the fraction of matches.

print(f"Accuracy: {accuracy:.3f}")

Accuracy: 1.000


This prints `Accuracy: 1.000` — a perfect score, because the threshold was
chosen from a gap that genuinely separates the classes with zero overlap,
not tuned after the fact to make the number look good.

## CustomRandomClassifier

A third custom classifier, `CustomRandomClassifier`, makes predictions with no
regard for the data at all — it predicts class 1 with probability `p` and
class 0 with probability `1-p`, purely by chance. Like `MajorityClassClassifier`,
this is a **baseline**: any real classifier trained on the features should be
expected to beat random guessing. Comparing a real model's accuracy against
this baseline (rather than just looking at the accuracy number alone) is a
common sanity check — if a model barely outperforms a coin flip, something is
likely wrong with the features, the labels, or the setup.

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.utils.multiclass import unique_labels
# Same imports as before — this version actually puts all four
# validation helpers to use inside fit() and predict().

class CustomRandomClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, p=0.5):
        self.p = p
        # Stores the probability of predicting class 1, same as before.

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        # Validates X and y together: checks they're proper array-like
        # inputs, converts them to NumPy arrays if needed, and confirms
        # X and y have the same number of rows. Raises a clear error
        # immediately if, say, X has 150 rows but y has 100 — instead
        # of failing later with a confusing shape mismatch elsewhere.

        self.classes_ = unique_labels(y)
        # Finds the distinct class labels present in y (e.g. array([0, 1]))
        # and stores them as self.classes_ — the attribute name
        # scikit-learn tooling conventionally expects a fitted classifier
        # to expose, so that other code (cross-validation, ensembling,
        # etc.) can discover which classes this model knows about.

        self.n_features_in_ = X.shape[1]
        # Records how many features X had during fit() — another
        # scikit-learn convention, used later to catch mismatched input
        # (e.g. calling predict() with a different number of columns
        # than the model was trained on).

        return self
        # As before, fit() doesn't need to learn any actual parameters
        # for this random baseline — but it now validates its inputs
        # and records the standard fitted-state attributes before
        # returning self.

    def predict(self, X):
        check_is_fitted(self, attributes=["classes_"])
        # Confirms fit() was already called by checking that
        # self.classes_ exists. If predict() is called on a fresh,
        # never-fitted instance, this raises a clear NotFittedError
        # instead of silently producing predictions from an unfitted
        # model.

        X = check_array(X)
        # Validates X the same way check_X_y did during fit() (proper
        # array-like input, no NaNs by default, correct shape),
        # converting it to a NumPy array if it isn't one already.

        random_preds = np.random.choice([0, 1], size=len(X), p=[1 - self.p, self.p])
        # Same random draw as before: one value from {0, 1} per row of
        # X, independently, using P(0) = 1-self.p and P(1) = self.p.

        return random_preds
        # Returns the array of random predictions, one per row of X.

In [ ]:
random_clf = CustomRandomClassifier(p=0.7)
# Creates an instance of CustomRandomClassifier with p=0.7, overriding
# the default of 0.5. This sets P(predict 1) = 0.7 and P(predict 0) = 0.3
# for every future call to predict() — no fitting has happened yet,
# this line only configures the object.
#
# Choosing p away from 0.5 is what makes this baseline useful for
# comparison on an imbalanced dataset: if the true positive rate in y
# is around 70%, a classifier that ignores the features entirely but
# guesses "1" 70% of the time will already score fairly high on plain
# accuracy. Setting p to roughly match the class balance is what makes
# this a fair baseline to beat, rather than one that's too easy
# (p=0.5 on an imbalanced target) or too hard (p picked arbitrarily).

In [ ]:
random_clf.fit(X, y_binary)
# Fits random_clf on the SAME binary target used for ThresholdClassifier
# (0 = setosa, 1 = not setosa), instead of the original 3-class y.
# check_X_y validates X and y_binary have matching row counts; then
# self.classes_ = unique_labels(y_binary) records exactly two classes,
# array([0, 1]) — satisfying CustomRandomClassifier's binary-only
# assumption, so fit() completes normally instead of raising the
# ValueError it would raise on the 3-class y.

CustomRandomClassifier(p=0.7)

In [ ]:
random_clf.predict(X)
# check_is_fitted confirms fit() already ran (it did, on X, y_binary).
# check_array validates X.
# np.random.choice(self.classes_, size=len(X), p=[1-self.p, self.p])
# draws one random prediction per row of X from self.classes_ —
# array([0, 1]), the two classes recorded during fit(binary) — using
# P(0) = 0.3, P(1) = 0.7 from p=0.7. X's actual feature values are
# never inspected; only its length (150 rows) is used.

array([1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1])

In [ ]:
np.random.seed(42)
# Fixes NumPy's global random number generator to a specific seed
# BEFORE either predict() call runs. Every subsequent call to
# np.random.choice() (used inside predict()) will now draw from the
# same reproducible sequence of "random" values — so re-running this
# notebook top-to-bottom produces the identical predictions and
# accuracy every time, instead of a new random result each run.

random_preds = random_clf.predict(X)
# First draw from the now-seeded random sequence.

random_accuracy = random_clf.score(X, y_binary)
# score() calls predict(X) again internally — a SECOND draw from the
# same seeded sequence, continuing where the first call left off. It
# is not a repeat of random_preds; it's the next values in the
# sequence, so random_preds and the predictions scored here are still
# two distinct random draws, just now both reproducible.

print(f"Random baseline accuracy (p=0.7): {random_accuracy:.3f}")
# Same as before — prints the accuracy to 3 decimal places.
# Prints the accuracy, formatted to 3 decimal places.

Random baseline accuracy (p=0.7): 0.573


## Additional Classifiers

Additional classifiers — `CustomRandomClassifier` (randomly predicts classes),
`HybridClassifier` (averages predictions from a Random Forest and an XGBoost
model), and `SklearnTFHybrid` (averages predictions from a Random Forest and a
simple PyTorch neural network) — are also defined.


### What Random Forest and XGBoost are

- **Random Forest** — an ensemble of many decision trees, each trained on a
  random subset of the data and features. Each tree votes on the class; the
  forest predicts whichever class gets the most votes (or, for
  `predict_proba()`, the average vote fraction per class). Training many
  varied trees and combining them reduces overfitting compared to a single
  deep tree.

- **XGBoost** ("Extreme Gradient Boosting") — also an ensemble of decision
  trees, but built differently: trees are added one at a time, each new tree
  trained specifically to correct the errors of the trees before it
  (gradient boosting). This sequential, error-correcting process often gives
  higher accuracy than Random Forest, at the cost of being more sensitive to
  hyperparameters and slower to train.

Both are tree-based ensembles and both natively implement scikit-learn's
`fit` / `predict` / `predict_proba` interface (via `RandomForestClassifier`
and `XGBClassifier`), which is why `HybridClassifier` can combine them
directly without needing a wrapper, unlike the PyTorch case.

### What "averaging predictions" actually means

Combining two models' outputs isn't a single well-defined operation — it
matters *what* is being averaged:

- **Averaging class labels directly** is awkward and usually wrong. If one
  model predicts class `0` and the other predicts class `1`, averaging the
  raw labels gives `0.5` — not a valid class, and meaningless for anything
  beyond binary 0/1 labels (it breaks entirely for 3+ classes, or for
  non-numeric labels like `"yes"`/`"no"`).
- **Averaging predicted probabilities** is what "averaging predictions"
  almost always means in practice. Each model exposes a `predict_proba()`
  method returning, for every sample, a probability per class (e.g.
  `[0.3, 0.7]` for a binary problem). The two models' probability vectors are
  averaged element-wise, and the final predicted class is whichever has the
  highest averaged probability. This is a standard ensembling technique
  often called **soft voting**.

So for both `HybridClassifier` and `SklearnTFHybrid`, expect `fit()` to call
each sub-model's own `.fit()`, and `predict()` to call each sub-model's
`predict_proba()`, average the results, then convert the averaged
probabilities into a final class label.

### Why mixing frameworks isn't just "plug two models in"

- **`HybridClassifier` (Random Forest + XGBoost)** is the more mechanical
  case: both `RandomForestClassifier` (scikit-learn) and `XGBClassifier`
  (XGBoost) already implement the same `fit` / `predict` / `predict_proba`
  interface scikit-learn expects. Averaging their `predict_proba()` outputs
  should be straightforward, with no extra glue code needed.

- **`SklearnTFHybrid` (Random Forest + PyTorch)** is a harder case. A raw
  PyTorch model is just a computational graph — it has no `.fit()`, no
  `.predict()`, no `.predict_proba()`. To combine it with a Random Forest the
  same way, the PyTorch side needs to be **wrapped**:
  - a `fit()`-equivalent that runs PyTorch's own training loop (forward pass,
    loss computation, backward pass, optimizer step, repeated over epochs)
  - a `predict_proba()`-equivalent that runs a forward pass in evaluation
    mode and passes the raw output logits through a `softmax` (multi-class)
    or `sigmoid` (binary) to turn them into probabilities that are actually
    comparable to the Random Forest's `predict_proba()` output

  Without that wrapping, there's no direct way to "average" a PyTorch
  model's output with a scikit-learn model's output — the two don't speak
  the same interface by default.


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
# RandomForestClassifier is scikit-learn's Random Forest implementation.
# xgb.XGBClassifier is XGBoost's classifier, built to follow the same
# fit/predict/predict_proba interface as scikit-learn models.

class HybridClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, sklearn_classifier=None, xgb_classifier=None):
        self.sklearn_classifier = sklearn_classifier
        self.xgb_classifier = xgb_classifier
        # Stores whatever was passed in, or None if nothing was. Using
        # None here (instead of RandomForestClassifier()/xgb.XGBClassifier()
        # directly in the signature) avoids the mutable-default-argument
        # trap: default arguments are evaluated ONCE when the class is
        # defined, so a mutable default like a model instance would be
        # silently shared and reused across every HybridClassifier
        # created without explicit arguments.

    def fit(self, X, y):
        self.sklearn_classifier_ = self.sklearn_classifier or RandomForestClassifier()
        self.xgb_classifier_ = self.xgb_classifier or xgb.XGBClassifier()
        # Builds a FRESH default model here, inside fit(), only if none
        # was provided at construction time. Each call to fit() on a
        # new instance gets its own model objects, never one shared
        # across instances. The trailing underscore (sklearn_classifier_,
        # not sklearn_classifier) follows scikit-learn's convention for
        # attributes that only exist after fitting.

        self.sklearn_classifier_.fit(X, y)
        self.xgb_classifier_.fit(X, y)
        # Trains both sub-models independently on the same X, y.

        return self

    def predict(self, X):
        sklearn_proba = self.sklearn_classifier_.predict_proba(X)
        xgb_proba = self.xgb_classifier_.predict_proba(X)
        # Gets each sub-model's predicted PROBABILITY per class for
        # every row of X, instead of a single predicted label. Shape:
        # (n_samples, n_classes) — e.g. [0.1, 0.7, 0.2] for a 3-class
        # problem, one row per sample.

        avg_proba = (sklearn_proba + xgb_proba) / 2
        # Averages the two probability arrays element-wise — the
        # correct notion of "averaging predictions" for a classifier,
        # since probabilities are directly comparable and combinable
        # across models, unlike raw class labels.

        return avg_proba.argmax(axis=1)
        # For each row, finds the INDEX of the highest averaged
        # probability — which is exactly the predicted class, since
        # predict_proba()'s columns are ordered to match the model's
        # class labels. This works correctly for any number of
        # classes, unlike averaging labels directly.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Splits X, y into an 80% training portion and a 20% held-out test
# portion. random_state=42 makes the split reproducible; stratify=y
# preserves the original 3-class proportions in both X_train/y_train
# and X_test/y_test, rather than risking an uneven split across species.

In [ ]:
hyb_clf = HybridClassifier()
# Creates a HybridClassifier instance with no arguments, so
# sklearn_classifier and xgb_classifier are both set to None at this
# point — self.sklearn_classifier_ / self.xgb_classifier_ (the actual
# RandomForestClassifier and XGBClassifier objects) don't exist yet,
# since those are only created inside fit(), not __init__().
#
# No training has happened yet — this line only configures the object,
# ready for hyb_clf.fit(X, y) next.

In [ ]:
hyb_clf.xgb_classifier
# Accesses the xgb_classifier attribute set in __init__(). Since
# hyb_clf was created with no arguments, this is still None at this
# point — fit() hasn't been called yet, so self.xgb_classifier_ (the
# actual fitted XGBClassifier, with the trailing underscore) doesn't
# exist. This line would display None as the cell's output.
#
# Note the naming distinction this relies on: xgb_classifier (no
# underscore) is the constructor argument, which stays None here;
# xgb_classifier_ (trailing underscore) is the fitted model created
# inside fit() — accessing that one now, before fit() has run, would
# raise an AttributeError instead of returning None.

In [ ]:
hyb_clf.fit(X_train, y_train)
# Fits both sub-models (Random Forest and XGBoost) using ONLY the
# training portion, X_train/y_train, from the earlier train_test_split
# — not the full X, y. This replaces the earlier hyb_clf.fit(X, y)
# call, so the test portion (X_test, y_test) stays unseen by either
# model during training.
#
# Random Forest and XGBoost both natively support multiclass problems,
# so no restriction like CustomRandomClassifier's binary-only check is
# needed here — predict_proba() will return one probability column per
# class (3, for the 3-species y_train) for each sub-model, and the
# soft-voting fix (average the two probability arrays, then argmax)
# works correctly regardless of how many classes are present.

HybridClassifier()

In [ ]:
hyb_clf_preds = hyb_clf.predict(X_test)
hyb_clf_accuracy = hyb_clf.score(X_test, y_test)
# Predicts on X_test — the held-out portion from train_test_split,
# which neither sub-model saw during hyb_clf.fit(X_train, y_train) —
# instead of the full X. score() internally calls predict(X_test)
# again, compares those predictions to y_test, and returns the
# fraction correct: a genuine measure of how well the model
# generalizes, not how well it memorized its own training data.

print(f"HybridClassifier test accuracy: {hyb_clf_accuracy:.3f}")

HybridClassifier test accuracy: 0.933


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
# torch/torch.nn/torch.optim: PyTorch's core building blocks — nn for
# defining network layers, optim for gradient-based optimizers.
# StandardScaler: scikit-learn's z-score scaler, used to standardize
# features before feeding them into the neural network.

class SimpleNN(nn.Module):
    def __init__(self, n_features=4, n_classes=3, hidden=64):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(n_features, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)
        # Two fully-connected layers: n_features inputs -> hidden units
        # -> n_classes outputs, all configurable rather than fixed
        # numbers, so the same class works for any dataset shape.

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x
        # Passes input through fc1, applies ReLU, then fc2, returning
        # raw, unnormalized logits — one score per class. No softmax
        # is applied here; that happens separately at inference time.

class SklearnTorchHybrid(BaseEstimator, ClassifierMixin):
    def __init__(self, sklearn_classifier=None, hidden=64, epochs=10, lr=0.001):
        self.sklearn_classifier = sklearn_classifier
        self.hidden = hidden
        self.epochs = epochs
        self.lr = lr
        # Stores exactly the arguments passed in, and nothing else.
        # sklearn_classifier defaults to None, so a fresh Random Forest
        # is created for each new instance rather than one object being
        # shared across every instance that omits this argument.

    def fit(self, X, y):
        self.sklearn_classifier_ = self.sklearn_classifier or RandomForestClassifier()
        self.sklearn_classifier_.fit(X, y)
        # Creates a Random Forest (the provided one, or a fresh default)
        # and trains it on X, y.

        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X)
        # Fits a scaler on the training data and stores it, so the
        # exact same mean/std can be reapplied to any data passed into
        # predict() later.

        n_features = X.shape[1]
        n_classes = len(np.unique(y))
        self.torch_model_ = SimpleNN(n_features=n_features, n_classes=n_classes, hidden=self.hidden)
        self.criterion_ = nn.CrossEntropyLoss()
        self.optimizer_ = optim.Adam(self.torch_model_.parameters(), lr=self.lr)
        # Builds the neural network sized to the actual data (number
        # of features and number of distinct classes in y), along with
        # a cross-entropy loss and an Adam optimizer for training it.

        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.long)
        # Converts the scaled features and integer labels into PyTorch
        # tensors: float32 for the features, long for the labels, since
        # CrossEntropyLoss requires integer class indices in that dtype.

        self.torch_model_.train()
        # Puts the network in training mode.

        for epoch in range(self.epochs):
            self.optimizer_.zero_grad()
            outputs = self.torch_model_(X_tensor)
            loss = self.criterion_(outputs, y_tensor)
            loss.backward()
            self.optimizer_.step()
        # Standard training loop: clear old gradients, run a forward
        # pass to get logits, compute the loss against the true labels
        # (CrossEntropyLoss applies log-softmax internally), backpropagate
        # gradients, then update the network's weights.

        return self

    def predict_proba(self, X):
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        # Scales the input using the stored scaler and converts it to
        # a tensor, ready for a forward pass.

        self.torch_model_.eval()
        with torch.no_grad():
            torch_logits = self.torch_model_(X_tensor)
            torch_proba = torch.softmax(torch_logits, dim=1).numpy()
        # Puts the network in evaluation mode, disables gradient
        # tracking (no training happens here), runs a forward pass to
        # get logits, then applies softmax to turn them into class
        # probabilities that sum to 1 per row.

        sklearn_proba = self.sklearn_classifier_.predict_proba(X)
        # Gets the Random Forest's own predicted class probabilities
        # for the same input.

        avg_proba = (sklearn_proba + torch_proba) / 2
        return avg_proba
        # Averages the two models' probability arrays element-wise,
        # combining the Random Forest's and the neural network's
        # predictions into one probability distribution per row.

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)
        # Picks the class with the highest averaged probability for
        # each row, giving one final predicted label per sample.

## Custom Pipeline

A scikit-learn pipeline is created combining the CustomScaler transformer and HybridClassifier. This pipeline is then fitted on the training data and evaluated on the test data, showcasing how custom transformers and classifiers can be integrated into scikit-learn pipelines for streamlined model training and evaluation.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load data
iris = load_iris()
X, y = iris.data, iris.target
# Reloads Iris fresh, with X as the 4 features and y as the full
# 3-class target (0/1/2) — attribute-style access this time
# (iris.data, iris.target) rather than the dict-style data["data"]
# used earlier in the notebook; both return the same arrays.

## For simplicity, convert to a binary classification problem
#y_binary = (y == 0).astype(int)  # 1 if setosa, 0 otherwise
# Commented out: an alternative binary target (1 = setosa, 0 =
# everything else), left here but not currently used — y below stays
# the full 3-class target.

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Splits X, y into an 80% training set and a 20% held-out test set,
# with random_state=42 fixing which rows land in each split.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
# Pipeline chains a sequence of steps into one object with its own
# fit/predict interface. classification_report prints per-class
# precision, recall, and F1-score from predictions and true labels.

pipeline = Pipeline([
    ('scaler', CustomScaler()),
    ('classifier', HybridClassifier())
])
# Builds a two-step pipeline: 'scaler' (CustomScaler) runs first,
# 'classifier' (HybridClassifier) runs second. Each step is named so
# it can be referred to individually later, e.g. pipeline.named_steps['scaler'].

pipeline.fit(X_train, y_train)
# Fits the whole chain in one call. Internally: CustomScaler.fit_transform(X_train)
# runs first, learning self.mean_/self.std_ from X_train and returning
# the scaled features; that scaled output is then passed into
# HybridClassifier.fit(scaled_X_train, y_train), which trains its
# Random Forest and XGBoost sub-models on the scaled data.

predictions = pipeline.predict(X_test)
# Runs the same chain on X_test: CustomScaler.transform(X_test) scales
# the test features using the mean/std learned from X_train (not
# X_test's own statistics), then HybridClassifier.predict() runs on
# that scaled output to produce final class predictions.

print(classification_report(y_test, predictions))
# Compares predictions against the true test labels y_test, printing
# precision, recall, and F1-score for each class, plus overall accuracy.

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## Explaining the Result

          precision    recall  f1-score   support

       0       1.00      1.00      1.00        10
       1       1.00      1.00      1.00         9
       2       1.00      1.00      1.00        11

accuracy                           1.00        30


### Reading the table

- **Rows 0, 1, 2** are the three Iris species (setosa, versicolor, virginica).
  Each row's **precision** (of everything predicted as this class, what fraction
  actually was), **recall** (of everything that actually was this class, what
  fraction got predicted correctly), and **f1-score** (their harmonic mean)
  all come out to **1.00** — every single sample in the test set was classified
  correctly, with no false positives and no false negatives for any class.
- **support** is how many test samples belonged to each class: 10 + 9 + 11 = 30,
  matching the `test_size=0.2` split of 150 total Iris samples (150 × 0.2 = 30).
- **accuracy = 1.00** confirms it directly: all 30 test samples were classified
  correctly.
- **macro avg** (unweighted mean across classes) and **weighted avg** (mean
  weighted by support) both equal 1.00 as well — with every class already at a
  perfect score, there's no difference for either average to reveal.

### Why a perfect score here isn't a red flag

A 100% accuracy result is usually worth being suspicious of — it often signals
data leakage or a bug (e.g. accidentally evaluating on training data). Here,
though, it's explainable without any of that:

- **The pipeline correctly separates fitting from evaluation.** `pipeline.fit(X_train, y_train)`
  and `pipeline.predict(X_test)` operate on the same train/test split used for
  `hyb_clf` earlier in the notebook — `X_test` is genuinely unseen data, not a
  re-evaluation of the training set.
- **Iris is an unusually easy, well-separated dataset.** The `ThresholdClassifier`
  section earlier already showed one feature (petal length) perfectly separating
  setosa from the other two species with zero overlap. The other two species
  are also fairly separable, just not with a single hard cutoff on one feature —
  which is exactly the kind of pattern a Random Forest + XGBoost ensemble
  (`HybridClassifier`) is well suited to pick up.
- **The test set is small (30 samples).** With so few test points, a strong
  model on an easy dataset landing on 0 mistakes is plausible by chance alone,
  not just a sign of a powerful pipeline.

### What this result demonstrates

The main point of this section — that a custom transformer (`CustomScaler`)
and a custom classifier (`HybridClassifier`) can be dropped into a standard
scikit-learn `Pipeline` and behave exactly like built-in components — is
confirmed: `fit()` correctly chained scaling into classification, `predict()`
correctly reused the training set's learned statistics on the test set, and
`classification_report` was able to evaluate the result exactly as it would
for any built-in scikit-learn model.